In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U transformers accelerate datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.4 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers accelerate torch librosa soundfile sentencepiece

In [ ]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "openai/whisper-large-v3"

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

model.eval()

print("✅ Model loaded and ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Model loaded and ready


In [ ]:
import torch
from transformers import AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

processor = AutoProcessor.from_pretrained("openai/whisper-large-v2")
print("✅ Processor loaded")

Device: cuda


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Processor loaded


In [ ]:
!pip uninstall -y peft transformers accelerate datasets torchaudio
!pip install --no-cache-dir \
  transformers==4.35.2 \
  accelerate==0.24.1 \
  datasets==2.14.6 \
  torchaudio \
  jiwer

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: datasets 4.6.1
Uninstalling datasets-4.6.1:
  Successfully uninstalled datasets-4.6.1
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 14.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 171.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 397.4 MB/s eta 0:00:

In [ ]:
# =====================================================
# WHISPER LARGE V3 - FINAL STABLE TAMIL TRANSCRIBER
# =====================================================

import os
import re
import gc
import torch
import librosa
import soundfile as sf
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# ======================
# PATHS (UNCHANGED)
# ======================
MODEL_NAME = "openai/whisper-large-v3"
DATA_PATH = "/content/drive/MyDrive/TamilDialect/Test"
OUTPUT_FILE = "/content/drive/MyDrive/TamilDialect/outputs/final_output.txt"

# ======================
# GPU CLEANUP
# ======================
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# ======================
# LOAD MODEL
# ======================
print("Loading Whisper model...")

processor = WhisperProcessor.from_pretrained(MODEL_NAME)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    low_cpu_mem_usage=True
).to(device)

model.eval()
print("✅ Whisper Large v3 loaded")

# ======================
# AUDIO LOADER (FAST + SAFE)
# ======================
def load_audio(path):
    try:
        audio, sr = librosa.load(path, sr=16000, mono=True)
    except Exception:
        audio, sr = sf.read(path)
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

    if audio is None or len(audio) < 1600:
        raise ValueError("Empty or invalid audio")

    audio = audio.astype(np.float32)
    audio = audio / (np.max(np.abs(audio)) + 1e-7)
    return audio

def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

# ======================
# PROCESS FILES
# ======================
files = sorted([f for f in os.listdir(DATA_PATH) if f.lower().endswith(".wav")])
print("Found files:", len(files))

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for wav in files:
        try:
            path = os.path.join(DATA_PATH, wav)
            audio = load_audio(path)

            inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(device)

            if device == "cuda":
                input_features = input_features.half()

            with torch.no_grad():
                predicted_ids = model.generate(
                    input_features,
                    forced_decoder_ids=processor.get_decoder_prompt_ids(
                        language="ta",
                        task="transcribe"
                    ),
                    max_new_tokens=256,
                    num_beams=3,              # Better Tamil accuracy
                    repetition_penalty=1.1,
                    no_repeat_ngram_size=3
                )

            text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
            text = clean_text(text)

            print(f"{wav} => {text}")
            out.write(f"{wav}|{text}\n")

            del audio, inputs, input_features, predicted_ids
            if device == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(f"{wav} => SKIPPED ({str(e)[:60]})")
            out.write(f"{wav}|\n")

print("\n✅ DONE")
print("Saved to:", OUTPUT_FILE)

Loading Whisper model...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


✅ Whisper Large v3 loaded
Found files: 579
test_0001.wav => பெய்யம் ஒரு பையன் ஏத்து படிக்கலாம் பாலமுர்கன் பெறு
test_0002.wav => நான் பீடு சுத்துவேன். தம்பி பார்க்கலாம்.
test_0003.wav => வணக்கம் அல்லாம் நான்தானே பார்க்கவேண்டும்
test_0004.wav => சுந்தக்காரங்கள் அன்னவுடையுள்ளா பக்த்தில் தானே இருக்கவேண்டும் என்னை விடிப்பார்க்களால் அல்லது அனைவருடன் சுத்நக் காரணங்க கொஞ்சம் இருப்கிறார். மற்றவிட அவருகின் குறையாளர்களும், வெளியூருடல்தான் இரு.
test_0005.wav => நான் ஜெரியாவங்க எதுவும் பங்சன் வைச்சா என்னை உங்களைக் கூப்புவதற்காண்டி வருவார்கள் இல்லை ஏதோம் தொலைபேசியில் அவர் போன்று சொல்வாங் க நாஙலம் செல்லுவோம்.
test_0006.wav => என் பையனுக்குப் பிடித்தது பிரியாணிதான். வாரத் தொரு ஒரு நாள் எப்படியோ செய்துகொண்டுகிறோம்.
test_0007.wav => ஒரு நாளை அவர்களுக்கு நேரம் கிடைத்தால் என்னுடைய வேலையை செய்வார்கள்.
test_0008.wav => எது பிடிக்கும்னா சாம்பார் தான் அப்படி விடுகிறோம் வாருங்கள் சா ம்பர் அவ்வையில் வைத்தா பிரியும்.
test_0009.wav => நான் இந்திரில் மீன் வப்போம் பிறகு சிக்கன் தான், அடிகடி வகியது மட்டா நீ ஏற்புமாவ